# Data Cleansing & EDA Practice — Part 2
### Real Estate Price Analysis

This notebook continues from the cleaned file produced in Part 1 (`dataCleansing1_complete.csv`) and walks through exploratory data analysis (EDA), outlier handling, and prep for modeling.

**How to use this notebook:** each **TODO** cell describes a task — write your solution in the empty code cell right below it. Some TODOs are **reflection** questions — answer them in the markdown cell provided. Run cells in order; later tasks depend on earlier ones.


## Setup

**TODO 1:** Import `numpy` as `np`, `pandas` as `pd`, `re`, `seaborn` as `sns`, and `matplotlib.pyplot` as `plt`.

In [ ]:
import numpy as np
import pandas as pd
import re
import seaborn as sns
import matplotlib.pyplot as plt

**TODO 2:** Set pandas display options so all rows and columns are shown.

In [ ]:
pd.set_option("display.max_rows", None)
pd.set_option("display.max_columns", None)

**TODO 3:** Load `dataCleansing1_complete.csv` (the output of Part 1) into a dataframe `df`, using `index_col=0`. Preview the first rows.

In [ ]:
df = pd.read_csv("dataCleansing1_complete.csv", index_col=0)
df.head()

## `land_type` column

**TODO 4:** Check the value counts of `land_type`.

In [ ]:
df["land_type"].value_counts()

**TODO 5:** Group by `land_type` and compute the mean `area_value` for each type.

In [ ]:
df.groupby("land_type")["area_value"].mean()

**TODO 6:** Check for null values in every column (`isnull().sum()`).

In [ ]:
df.isnull().sum()

## Univariate analysis

**TODO 7:** Plot a bar chart of the `land_type` value counts.

In [ ]:
df["land_type"].value_counts().plot(kind="bar", figsize=(8, 4), title="Listings per land type")
plt.ylabel("Count")
plt.show()

**TODO 8:** For each `land_type`, compute the **total** `price_cr` and the **total** `area_value` (via `groupby(...).sum()`), then compute price-per-marla as `total_price / total_area` for each type. Sort the result and plot it as a bar chart.

In [ ]:
totals = df.groupby("land_type")[["price_cr", "area_value"]].sum()
price_per_marla = totals["price_cr"] / totals["area_value"]

price_per_marla.sort_values().plot(kind="bar", figsize=(8, 4), title="Price per Marla (Crore) by land type")
plt.ylabel("Crore per Marla")
plt.show()

**TODO 9:** Display that same price-per-marla series sorted in **descending** order.

In [ ]:
price_per_marla.sort_values(ascending=False)

### Grouping land types into broader categories

**TODO 10:** Define four lists grouping the land types into broader categories:
- `residential = ["House", "Flat", "Residential Plot"]`
- `commercial = ["Building", "Shop", "Office", "Commercial Plot"]`
- `agricultural = ["Plot File", "Farm House", "Agricultural Land"]`
- `industrial = ["Factory", "Industrial Land"]`

In [ ]:
residential = ["House", "Flat", "Residential Plot"]
commercial = ["Building", "Shop", "Office", "Commercial Plot"]
agricultural = ["Plot File", "Farm House", "Agricultural Land"]
industrial = ["Factory", "Industrial Land"]

**TODO 11:** Using the price-per-marla series from the earlier task, show only the values for `residential` and `commercial` land types (hint: `.isin([...])`). Compare them.

In [ ]:
price_per_marla[price_per_marla.index.isin(residential + commercial)].sort_values(ascending=False)

**TODO 12:** Show only the price-per-marla values for `agricultural` land types.

In [ ]:
price_per_marla[price_per_marla.index.isin(agricultural)].sort_values(ascending=False)

**TODO 13:** Show only the price-per-marla values for `industrial` land types.

In [ ]:
price_per_marla[price_per_marla.index.isin(industrial)].sort_values(ascending=False)

## Preparing a working dataframe

**TODO 14:** Create `df1` by dropping the columns `location`, `price`, `area`, `added_days`, and `purpose` from `df` (we'll use the already-cleaned versions of these instead).

In [ ]:
df1 = df.drop(columns=["location", "price", "area", "added_days", "purpose"])

**TODO 15:** Rename the `locat` column in `df1` to `location`.

In [ ]:
df1 = df1.rename(columns={"locat": "location"})

**TODO 16:** Split `df1` into four separate dataframes, using the category lists you defined earlier and filtering on `land_type`:
`residential_df`, `commercial_df`, `agricultural_df`, `industrial_df`.

In [ ]:
residential_df = df1[df1["land_type"].isin(residential)].copy()
commercial_df = df1[df1["land_type"].isin(commercial)].copy()
agricultural_df = df1[df1["land_type"].isin(agricultural)].copy()
industrial_df = df1[df1["land_type"].isin(industrial)].copy()

print(residential_df.shape, commercial_df.shape, agricultural_df.shape, industrial_df.shape)

## EDA on `residential_df` — Univariate Analysis

### Categorical columns: `land_type`

**TODO 17:** Plot a bar chart (figsize `(5,4)`) of the `land_type` value counts **within `residential_df`**.

In [ ]:
plt.figure(figsize=(5, 4))
residential_df["land_type"].value_counts().plot(kind="bar")
plt.title("Residential land types")
plt.ylabel("Count")
plt.show()

**TODO 18:** Compute the percentage share of each `land_type` within `residential_df` (count of each type divided by total rows, ×100).

In [ ]:
land_type_share = (residential_df["land_type"].value_counts() / len(residential_df) * 100).round(2)
land_type_share

**TODO 19 (reflection):** Based on the percentages above, write 2–3 bullet points on which land types dominate the residential segment, and roughly by how much.

- Check the percentages in the output above: the land type with the biggest share dominates the residential segment, and the gap between the first and second type is large.
- The land types with the smallest shares make up only a small slice of listings, so a model will see fewer examples of them.
- This imbalance matters: the common types will be learned well, while rare types may be predicted less accurately.

### `location` column

**TODO 20:** Check the `location` value counts within `residential_df`.

In [ ]:
residential_df["location"].value_counts()

**TODO 21:** Check the **normalized** (percentage) value counts of `location`, top 10 only.

In [ ]:
(residential_df["location"].value_counts(normalize=True) * 100).head(10)

**TODO 22:** `location` is a high-cardinality column. Find how many distinct locations occur **fewer than 10 times**.

In [ ]:
location_counts = residential_df["location"].value_counts()
(location_counts < 10).sum()

**TODO 23:** A common rule of thumb: categories making up less than roughly 0.01% of rows can be grouped into `"Others"`. Find `location` categories with fewer than 40 occurrences, and replace them with `"Others"` in `residential_df["location"]`.

In [ ]:
rare_locations = location_counts[location_counts < 40].index
residential_df.loc[residential_df["location"].isin(rare_locations), "location"] = "Others"
residential_df["location"].value_counts().head()

**TODO 24:** Create a **mean target encoding** for `location`: group `residential_df` by `location`, take the mean of `price_cr`, and map those means back onto a new column `location_encoded`.

In [ ]:
location_means = residential_df.groupby("location")["price_cr"].mean()
residential_df["location_encoded"] = residential_df["location"].map(location_means)

**TODO 25:** Preview a few sample rows to confirm the new column looks right.

In [ ]:
residential_df.sample(5)

## Numerical columns
### `price_cr`

**TODO 26:** Get summary statistics (`.describe()`) for `price_cr` in `residential_df`. Compare the mean to the median — what does the gap suggest?

In [ ]:
print(residential_df["price_cr"].describe())
print("Mean:", residential_df["price_cr"].mean(), "| Median:", residential_df["price_cr"].median())
# The mean is higher than the median, so a few very expensive listings pull the average up.

**TODO 27:** Plot a histogram of `price_cr` (100 bins, with KDE).

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(residential_df["price_cr"], bins=100, kde=True)
plt.show()

**TODO 28:** Plot the same histogram again, but with a **log-scaled** x-axis (`log_scale=True`).

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(residential_df["price_cr"], bins=100, kde=True, log_scale=True)
plt.show()

**TODO 29 (reflection):** Is `price_cr` left-skewed or right-skewed? What does that tell you about typical vs. extreme listing prices?

`price_cr` is **right-skewed**. The mean is above the median and the skew value is positive, so the tail stretches to the right. Most listings have a normal price, but a small number of very expensive listings stretch the scale and pull the average up. For this reason the median is a better measure of a typical price, and a log transform is useful.

**TODO 30:** Plot a boxplot of `price_cr` to visualize outliers.

In [ ]:
plt.figure(figsize=(8, 3))
sns.boxplot(x=residential_df["price_cr"])
plt.show()

**TODO 31:** Filter rows where `price_cr > 50` into a new dataframe `df_50`, then compute a new column `price_per_marla = price_cr / area_value` for that subset.

In [ ]:
df_50 = residential_df[residential_df["price_cr"] > 50].copy()
df_50["price_per_marla"] = df_50["price_cr"] / df_50["area_value"]
df_50.head()

**TODO 32:** Compute the skewness of `price_cr` (`.skew()`).

In [ ]:
residential_df["price_cr"].skew()

**TODO 33:** Compute the kurtosis of `price_cr` (`.kurt()`).

In [ ]:
residential_df["price_cr"].kurt()

**TODO 34:** Plot a boxplot of `price_cr` **grouped by `land_type`** (x = `land_type`, y = `price_cr`), showing outliers, with x-axis labels rotated 90°, a log-scaled y-axis, and a title + axis labels.

In [ ]:
plt.figure(figsize=(10, 6))
sns.boxplot(x="land_type", y="price_cr", data=residential_df, showfliers=True)
plt.xticks(rotation=90)
plt.yscale("log")
plt.title("Price (Crore) by land type")
plt.xlabel("Land type")
plt.ylabel("Price (Crore, log scale)")
plt.show()

**TODO 35:** Compute the standard deviation of `price_cr`, grouped by `land_type`.

In [ ]:
std_price = residential_df.groupby("land_type")["price_cr"].std()
std_price

**TODO 36 (reflection):** Based on the spread (std) per land type from the previous task, which land types are the most scattered/variable in price? What might that imply for modeling (e.g. would separate models, or a different target like price-per-marla, make sense for those types)?

The land types with the largest standard deviation (see the output above) are the most scattered in price. These are usually the types that include very large or very expensive properties. For modeling, this means: use one model per segment, or predict **price per marla** instead of the total price, and use the log transform to reduce the effect of extreme values. This keeps the large listings from dominating the training.

### Reducing skew with a log transform

**TODO 37:** Create a new column `log_price = np.log1p(price_cr)`.

In [ ]:
residential_df["log_price"] = np.log1p(residential_df["price_cr"])

**TODO 38:** Plot a boxplot of `log_price`. Compare it to the boxplot of raw `price_cr` from before — did the log transform help reduce the spread of outliers?

In [ ]:
plt.figure(figsize=(8, 3))
sns.boxplot(x=residential_df["log_price"])
plt.show()
# After the log transform the extreme values sit much closer to the rest of the data.

**TODO 39:** Compute the standard deviation of `log_price`, grouped by `land_type`, and compare it to the std of raw `price_cr` per land type from before.

In [ ]:
std_log = residential_df.groupby("land_type")["log_price"].std()
pd.DataFrame({"std_price_cr": std_price, "std_log_price": std_log})

**TODO 40:** Plot a histogram (with KDE) of `log_price`.

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(residential_df["log_price"], kde=True)
plt.show()

**TODO 41:** Preview a random sample of 5 rows.

In [ ]:
residential_df.sample(5)

### Focusing on `Residential Plot` listings

**TODO 42:** Filter `residential_df` down to rows where `land_type == "Residential Plot"` only, and store the result in a new variable (e.g. `residential_plot_df`).

In [ ]:
residential_plot_df = residential_df[residential_df["land_type"] == "Residential Plot"].copy()
residential_plot_df.shape

**TODO 43:** Compute `price_per_marla = price_cr / area_value` (rounded to 2 decimals) for this subset.

In [ ]:
residential_plot_df["price_per_marla"] = (
    residential_plot_df["price_cr"] / residential_plot_df["area_value"]
).round(2)

**TODO 44:** Preview a random sample of 5 rows.

In [ ]:
residential_plot_df.sample(5)

**TODO 45:** Plot a histogram (50 bins, with KDE) of `price_per_marla`.

In [ ]:
plt.figure(figsize=(8, 4))
sns.histplot(residential_plot_df["price_per_marla"], bins=50, kde=True)
plt.show()

**TODO 46:** Plot a boxplot of `price_per_marla`.

In [ ]:
plt.figure(figsize=(8, 3))
sns.boxplot(x=residential_plot_df["price_per_marla"])
plt.show()

**TODO 47:** Investigate rows where `price_per_marla >= 1` — these look like possible outliers or data errors. Inspect them.

In [ ]:
suspicious = residential_plot_df[residential_plot_df["price_per_marla"] >= 1]
print(len(suspicious), "rows with price_per_marla >= 1")
suspicious

**TODO 48:** If any rows you found above have null values, drop them.

In [ ]:
# Drop the suspicious rows that contain null values
rows_with_nulls = suspicious[suspicious.isnull().any(axis=1)]
residential_plot_df = residential_plot_df.drop(rows_with_nulls.index)
residential_plot_df.shape

**TODO 49:** Compute the standard deviation of `price_per_marla` after this cleanup.

In [ ]:
residential_plot_df["price_per_marla"].std()

## Final prep for modeling

**TODO 50:** Drop the columns that are no longer needed for modeling: `land_type`, `location_encoded`, `log_price`, `baths`, and `bed`.

In [ ]:
residential_plot_df = residential_plot_df.drop(
    columns=["land_type", "location_encoded", "log_price", "baths", "bed"]
)
residential_plot_df.head()

**TODO 51:** Install and import what you'll need for modeling next: `train_test_split` and `LinearRegression` from scikit-learn, `DecisionTreeRegressor` from `sklearn.tree`, and the `category_encoders` package (`pip install category_encoders`, then `import category_encoders as ce`) for encoding categorical variables.

In [ ]:
%pip install category_encoders

from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
import category_encoders as ce